# 🚀 LLM Simple - Compilation Tout-en-Un v2

**Version avec compilation minimale llama.cpp**

**⏱️ Temps: 12-15 minutes**

**Runtime → Run all** et c'est tout! 🎉

In [ ]:
# Cellule 1: Installation
print("🔧 Installation...\n")
!apt-get update -qq
!apt-get install -y -qq mingw-w64 cmake ninja-build zip unzip file wget git
!pip install -q scons
print("\n✅ Outils installés!")
!x86_64-w64-mingw32-g++ --version | head -1

In [ ]:
# Cellule 2: Créer fichiers C++
import os
print("📝 Création fichiers...\n")

!mkdir -p /content/{llm_simple/{include,src},addons/llm_simple/bin}

# llm_simple.h
with open("/content/llm_simple/include/llm_simple.h", "w") as f:
    f.write("""#pragma once
#include <godot_cpp/classes/node.hpp>
#include <godot_cpp/core/class_db.hpp>
#include <godot_cpp/variant/utility_functions.hpp>

struct llama_model;
struct llama_context;
struct llama_sampler;

namespace godot {
class LLMSimple : public Node {
    GDCLASS(LLMSimple, Node)
private:
    llama_model* model;
    llama_context* ctx;
    llama_sampler* sampler;
    int n_ctx;
    bool is_loaded;
    String last_error;
protected:
    static void _bind_methods();
public:
    LLMSimple();
    ~LLMSimple();
    bool load_model(const String& model_path, int context_size = 8192);
    String generate(const String& prompt, int max_tokens = 256);
    void unload_model();
    bool is_model_loaded() const { return is_loaded; }
    String get_last_error() const { return last_error; }
    int get_context_size() const { return n_ctx; }
};
}
""")

# llm_simple.cpp
with open("/content/llm_simple/src/llm_simple.cpp", "w") as f:
    f.write('''#include "llm_simple.h"
#include <godot_cpp/core/class_db.hpp>
#include <godot_cpp/classes/project_settings.hpp>
#include "llama.h"
#include <vector>
using namespace godot;
void LLMSimple::_bind_methods() {
    ClassDB::bind_method(D_METHOD("load_model", "model_path", "context_size"), &LLMSimple::load_model, DEFVAL(8192));
    ClassDB::bind_method(D_METHOD("generate", "prompt", "max_tokens"), &LLMSimple::generate, DEFVAL(256));
    ClassDB::bind_method(D_METHOD("unload_model"), &LLMSimple::unload_model);
    ClassDB::bind_method(D_METHOD("is_model_loaded"), &LLMSimple::is_model_loaded);
    ClassDB::bind_method(D_METHOD("get_last_error"), &LLMSimple::get_last_error);
    ClassDB::bind_method(D_METHOD("get_context_size"), &LLMSimple::get_context_size);
    ADD_SIGNAL(MethodInfo("generation_started"));
    ADD_SIGNAL(MethodInfo("generation_finished", PropertyInfo(Variant::STRING, "text")));
    ADD_SIGNAL(MethodInfo("generation_error", PropertyInfo(Variant::STRING, "error")));
}
LLMSimple::LLMSimple() { model=nullptr; ctx=nullptr; sampler=nullptr; n_ctx=8192; is_loaded=false; llama_backend_init(); }
LLMSimple::~LLMSimple() { unload_model(); llama_backend_free(); }
bool LLMSimple::load_model(const String& model_path, int context_size) {
    unload_model(); n_ctx=context_size;
    String abs_path = ProjectSettings::get_singleton()->globalize_path(model_path);
    llama_model_params mp = llama_model_default_params(); mp.n_gpu_layers=0;
    model = llama_load_model_from_file(abs_path.utf8().get_data(), mp);
    if(!model) { last_error="Failed to load model"; return false; }
    llama_context_params cp = llama_context_default_params(); cp.n_ctx=n_ctx; cp.n_batch=512; cp.n_threads=4;
    ctx = llama_new_context_with_model(model, cp);
    if(!ctx) { last_error="Failed to create context"; llama_free_model(model); model=nullptr; return false; }
    sampler = llama_sampler_chain_init(llama_sampler_chain_default_params());
    llama_sampler_chain_add(sampler, llama_sampler_init_temp(0.7f));
    llama_sampler_chain_add(sampler, llama_sampler_init_top_k(50));
    llama_sampler_chain_add(sampler, llama_sampler_init_top_p(0.95f,1));
    llama_sampler_chain_add(sampler, llama_sampler_init_dist(1234));
    is_loaded=true; return true;
}
String LLMSimple::generate(const String& prompt, int max_tokens) {
    if(!is_loaded) return "";
    std::vector<llama_token> toks(n_ctx);
    int n = llama_tokenize(model, prompt.utf8().get_data(), prompt.length(), toks.data(), toks.size(), true, false);
    if(n<0) return "";
    toks.resize(n);
    llama_sampler_reset(sampler);
    if(llama_decode(ctx, llama_batch_get_one(toks.data(),n))!=0) return "";
    String res;
    for(int i=0;i<max_tokens;i++){
        llama_token t = llama_sampler_sample(sampler,ctx,-1);
        if(llama_token_is_eog(model,t)) break;
        char b[256]; int m=llama_token_to_piece(model,t,b,sizeof(b),0,false);
        if(m<0) break;
        res+=String::utf8(b,m);
        if(llama_decode(ctx,llama_batch_get_one(&t,1))!=0) break;
    }
    return res;
}
void LLMSimple::unload_model() { if(sampler){llama_sampler_free(sampler);sampler=nullptr;} if(ctx){llama_free(ctx);ctx=nullptr;} if(model){llama_free_model(model);model=nullptr;} is_loaded=false; }
''')

# register_types
with open("/content/llm_simple/src/register_types.cpp", "w") as f:
    f.write('''#include "llm_simple.h"
#include <gdextension_interface.h>
#include <godot_cpp/godot.hpp>
using namespace godot;
void init(ModuleInitializationLevel l){if(l!=MODULE_INITIALIZATION_LEVEL_SCENE)return;ClassDB::register_class<LLMSimple>();}
void uninit(ModuleInitializationLevel l){}
extern "C" {
GDExtensionBool GDE_EXPORT llm_simple_library_init(GDExtensionInterfaceGetProcAddress p,const GDExtensionClassLibraryPtr l,GDExtensionInitialization*r){
    GDExtensionBinding::InitObject o(p,l,r); o.register_initializer(init); o.register_terminator(uninit);
    o.set_minimum_library_initialization_level(MODULE_INITIALIZATION_LEVEL_SCENE); return o.init();}
}''')

# .gdextension
with open("/content/addons/llm_simple/llm_simple.gdextension","w") as f:
    f.write("[configuration]\nentry_symbol=\"llm_simple_library_init\"\ncompatibility_minimum=\"4.2\"\n[libraries]\nwindows.release.x86_64=\"res://addons/llm_simple/bin/llm_simple.windows.template_release.x86_64.dll\"")

print("✅ Fichiers créés!")

In [ ]:
# Cellule 3: Télécharger dépendances + PATCH godot-cpp
import os
os.chdir("/content")

print("📥 Téléchargement...\n")

# godot-cpp
!git clone --quiet --depth 1 --branch 4.2 https://github.com/godotengine/godot-cpp.git
print("✅ godot-cpp\n")

# PATCH godot-cpp: Commenter TOUT le code std::mutex
print("🔧 Patch godot-cpp (mutex MinGW - COMPLET)...\n")

# Patch HEADER (class_db.hpp)
with open("/content/godot-cpp/include/godot_cpp/core/class_db.hpp","r") as f:
    h = f.read()

# Commenter la déclaration du mutex
h = h.replace("static std::mutex engine_singletons_mutex;", "// static std::mutex engine_singletons_mutex;")

# Commenter TOUTES les utilisations de std::lock_guard dans le header
h = h.replace("std::lock_guard<std::mutex> lock(engine_singletons_mutex);", "// std::lock_guard<std::mutex> lock(engine_singletons_mutex);")

with open("/content/godot-cpp/include/godot_cpp/core/class_db.hpp","w") as f:
    f.write(h)

# Patch SOURCE (class_db.cpp)
with open("/content/godot-cpp/src/core/class_db.cpp","r") as f:
    c = f.read()

# Commenter la définition du mutex
c = c.replace("std::mutex ClassDB::engine_singletons_mutex;", "// std::mutex ClassDB::engine_singletons_mutex;")

# Commenter toutes les utilisations de std::lock_guard dans le source
c = c.replace("std::lock_guard<std::mutex> lock(engine_singletons_mutex);", "// std::lock_guard<std::mutex> lock(engine_singletons_mutex);")

with open("/content/godot-cpp/src/core/class_db.cpp","w") as f:
    f.write(c)

print("✅ Header patché (déclaration + utilisations)")
print("✅ Source patché (définition + utilisations)")
print("✅ godot-cpp prêt!\n")

# llama.cpp
!git clone --quiet --depth 1 --branch b4357 https://github.com/ggerganov/llama.cpp.git
print("✅ llama.cpp\n")

In [ ]:
# Cellule 4: Compiler godot-cpp (7-8 min)
import os
os.chdir("/content/godot-cpp")
print("🔨 Compilation godot-cpp (7-8 min)...\n⏳ Patience...\n")
!scons platform=windows target=template_release arch=x86_64 use_mingw=yes -j$(nproc) 2>&1 | tail -30
if os.path.exists("bin/libgodot-cpp.windows.template_release.x86_64.a"):
    print("\n✅ godot-cpp OK!")
else:
    raise Exception("❌ godot-cpp compilation failed")

In [ ]:
# Cellule 5: Compiler llama.cpp (3-5 min - MODE MINIMAL)
import os
os.chdir("/content/llama.cpp")
print("🔨 Compilation llama.cpp (minimal, 3-5 min)...\n")

# CMake avec options minimales
!cmake -B build \
    -DCMAKE_TOOLCHAIN_FILE=/content/llama.cpp/cmake/x64-mingw-w64.cmake \
    -DCMAKE_BUILD_TYPE=Release \
    -DLLAMA_BUILD_TESTS=OFF \
    -DLLAMA_BUILD_EXAMPLES=OFF \
    -DLLAMA_BUILD_SERVER=OFF \
    -DGGML_CUDA=OFF \
    -DGGML_OPENMP=OFF \
    -DGGML_BLAS=OFF \
    -DGGML_NATIVE=OFF \
    -G Ninja

print("\n🔧 Build (seulement bibliothèques)...\n")
!cmake --build build --target llama ggml -j$(nproc) 2>&1 | tail -20

# Vérifier
if os.path.exists("build/src/libllama.a") and os.path.exists("build/ggml/src/libggml.a"):
    print("\n✅ llama.cpp OK!")
else:
    print("\n⚠️ Vérification fichiers...")
    !find build -name "*.a" -type f
    raise Exception("❌ llama.cpp compilation failed")

In [ ]:
# Cellule 6: Compiler llm_simple (1-2 min)
print("🔨 Compilation llm_simple...\n")

!x86_64-w64-mingw32-g++ -shared -o /content/addons/llm_simple/bin/llm_simple.windows.template_release.x86_64.dll \
    /content/llm_simple/src/llm_simple.cpp \
    /content/llm_simple/src/register_types.cpp \
    -I/content/llm_simple/include \
    -I/content/godot-cpp/include \
    -I/content/godot-cpp/gen/include \
    -I/content/llama.cpp/include \
    -I/content/llama.cpp/ggml/include \
    /content/godot-cpp/bin/libgodot-cpp.windows.template_release.x86_64.a \
    /content/llama.cpp/build/src/libllama.a \
    /content/llama.cpp/build/ggml/src/libggml.a \
    -static-libgcc -static-libstdc++ \
    -std=c++17 -O3 \
    -Wl,--allow-multiple-definition

import os
if os.path.exists("/content/addons/llm_simple/bin/llm_simple.windows.template_release.x86_64.dll"):
    !ls -lh /content/addons/llm_simple/bin/*.dll
    print("\n✅ DLL compilée!")
else:
    raise Exception("❌ DLL compilation failed")

In [ ]:
# Cellule 7: Package et téléchargement
from google.colab import files
import os
os.chdir("/content")

print("📦 Packaging...\n")

# README
with open("addons/llm_simple/README.txt","w") as f:
    f.write("""LLM SIMPLE pour Godot 4.2+

INSTALLATION:
1. Copier addons/llm_simple dans votre projet Godot
2. Télécharger un modèle GGUF (Qwen2.5-3B ou Phi-3-mini)
3. Placer dans models/ de votre projet
4. Redémarrer Godot

UTILISATION:
extends Node

var llm: LLMSimple

func _ready():
    llm = LLMSimple.new()
    add_child(llm)
    
    if llm.load_model("res://models/qwen2.5-3b.gguf", 8192):
        var response = llm.generate("Bonjour!", 256)
        print(response)

func _exit_tree():
    if llm:
        llm.unload_model()

MODÈLES RECOMMANDÉS:
- Qwen2.5-3B-Instruct (Q4): https://huggingface.co/Qwen/Qwen2.5-3B-Instruct-GGUF
- Phi-3-mini-4k (Q4): https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf

Fichier recommandé: qwen2.5-3b-instruct-q4_k_m.gguf (~2 GB)
""")

# Test script
with open("TestLLM.gd","w") as f:
    f.write("""extends Node

var llm: LLMSimple

func _ready():
    llm = LLMSimple.new()
    add_child(llm)
    
    print("🤖 Chargement modèle...")
    if llm.load_model("res://models/qwen2.5-3b.gguf", 8192):
        print("✅ Modèle chargé!")
        print("📝 Génération...")
        var response = llm.generate("Bonjour! Qui es-tu?", 100)
        print("🎭 Réponse: " + response)
    else:
        print("❌ Erreur: " + llm.get_last_error())

func _exit_tree():
    if llm:
        llm.unload_model()
        print("👋 Modèle déchargé")
""")

# ZIP final
!zip -r llm_simple_addon.zip addons/ TestLLM.gd

print("\n" + "="*60)
print("✅ COMPILATION RÉUSSIE!")
print("="*60)
print("\n📊 Contenu:")
!unzip -l llm_simple_addon.zip | head -15
print("\n📥 Téléchargement du ZIP...\n")

files.download("llm_simple_addon.zip")

print("\n🎉 TERMINÉ!\n")
print("Prochaines étapes:")
print("1. Extraire llm_simple_addon.zip")
print("2. Copier addons/llm_simple/ dans votre projet Godot")
print("3. Télécharger Qwen2.5-3B: https://huggingface.co/Qwen/Qwen2.5-3B-Instruct-GGUF")
print("4. Fichier: qwen2.5-3b-instruct-q4_k_m.gguf")
print("5. Placer dans models/qwen2.5-3b.gguf")
print("6. Utiliser TestLLM.gd pour tester!")
print("\n📚 Documentation: voir QWEN_3B_MODELS.md")